# Digital Signal Processing Training Program 

![Texto Alternativo](images/V-CC_Color_Completa_TextoAzul.png)

### Dev by:
-   Elmer Pimentel Farias


# Analog-to-Digital and Digital-to-Analog Conversors

## Introduction

In modern electronic systems, Analog-to-Digital Converters (ADCs) and Digital-to-Analog Converters (DACs) serve as the essential interface between the analog physical world and digital signal processing systems. This notebook explores the critical stages of this conversion process: **Sampling**, **Holding**, **Quantization**, and **Encoding**.

The conversion process begins with sampling, where a continuous signal is measured periodically to create discrete values in time. Following this, quantization maps these continuous amplitude values to a finite set of $L = 2^N$ levels, where $N$ is the number of bits. This process introduces an inherent quantization error ($n_q$), which can be modeled as uniform noise.

### 1. Quantization Step ($\Delta$)

The resolution of the quantizer, or the distance between two adjacent levels, is determined by the dynamic range and the number of bits ($N$):

$$\Delta = \frac{V_{max} - V_{min}}{2^N}$$


### 2. Quantization Noise Power ($P_q$)

Under the assumption that the error is uniformly distributed between $[-\frac{\Delta}{2}, \frac{\Delta}{2}]$, the average power of the quantization noise is given by its variance:

$$P_q = \sigma_{n_q}^2 = \frac{\Delta^2}{12}$$

### 3. Ideal Signal-to-Quantization-Noise Ratio (SQNR)

For an ideal $N$-bit ADC excited by a full-scale sine wave, the SQNR in decibels is expressed as:

$$SQNR_{ideal} = 6.02N + 1.76 \text{ [dB]}$$


* The 6,02 term represents the 6 dB gain in SNR for every additional bit.


* The 1,72 term is associated with the RMS power of a sinusoid.



### 4. Effective Number of Bits (ENOB)

In real-world scenarios where distortion and non-ideal noise are present, we measure the $SNR_{real}$ to calculate the effective resolution of the converter:


$$ENOB = \frac{SNR_{real} - 1.76}{6.02}$$


## Project Objectives

We will develop a Python environment to:

* **Simulate Sampling and Reconstruction:** Model the sampling of a square wave and its reconstruction via Zero-Order Hold (ZOH).


* **Implement Uniform Quantization:** Create a function to map analog values to discrete levels and calculate the resulting error $n_q = V - V_q$ .


* **Calculate SNR and ENOB:** Compute the power of the original signal versus the noise to evaluate if our simulated ADC meets its theoretical bit depth.


* **Spectral Analysis:** Use Fast Fourier Transforms (FFT) to generate spectrum graphs and identify the harmonic distortions introduced by quantization.



### 1- Bibliotecas Importadas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import freqz
from scipy import signal
from numba import njit
from scipy.linalg import toeplitz, inv
import pandas as pd
import numpy as np
import time


In [1]:
import numpy as np

# ==========================================
# 1. Parâmetros do Sinal e do ADC
# ==========================================
fs = 10000        # Taxa de amostragem em Hz
f_in = 100        # Frequência do sinal senoidal em Hz
N_bits = 12       # Resolução nominal (teórica) do ADC
V_max = 1.0       # Tensão máxima (1V)
V_min = -1.0      # Tensão mínima (-1V)

# Eixo de tempo (1 segundo de amostragem)
t = np.arange(0, 1, 1/fs)

# Criando o sinal analógico limpo (Senoide em escala plena / full-scale)
amplitude = (V_max - V_min) / 2
sinal_limpo = amplitude * np.sin(2 * np.pi * f_in * t)

# ==========================================
# 2. Simulando o ADC Real (Quantização + Ruído)
# ==========================================
# Calculando o passo de quantização (Delta)
delta = (V_max - V_min) / (2**N_bits)

# Adicionando um ruído térmico/eletrônico (típico de componentes reais)
# Vamos forçar um ruído um pouco maior que o ruído de quantização natural
potencia_ruido_extra = (delta * 1.5)**2 
ruido_eletrico = np.random.normal(0, np.sqrt(potencia_ruido_extra), len(t))

sinal_com_imperfeicoes = sinal_limpo + ruido_eletrico

# Simulando a quantização (arredondando para os níveis do ADC)
sinal_adc = np.round(sinal_com_imperfeicoes / delta) * delta

# ==========================================
# 3. Calculando o ENOB
# ==========================================
# O ruído total é a diferença entre o sinal digitalizado e o sinal ideal
erro_total = sinal_adc - sinal_limpo

# Calculando a potência do sinal e a potência do ruído (RMS)
rms_sinal = np.sqrt(np.mean(sinal_limpo**2))
rms_ruido = np.sqrt(np.mean(erro_total**2))

# Calculando a Relação Sinal-Ruído (SNR) em dB
snr_real_db = 20 * np.log10(rms_sinal / rms_ruido)

# Calculando o ENOB usando a fórmula matemática
enob = (snr_real_db - 1.76) / 6.02

# ==========================================
# 4. Exibindo os Resultados
# ==========================================
print(f"Resolução nominal do ADC: {N_bits} bits")
print(f"SNR Ideal Teórica:        {(6.02 * N_bits + 1.76):.2f} dB")
print(f"SNR Medida na prática:    {snr_real_db:.2f} dB")
print(f"ENOB Calculado:           {enob:.2f} bits")

Resolução nominal do ADC: 12 bits
SNR Ideal Teórica:        74.00 dB
SNR Medida na prática:    59.48 dB
ENOB Calculado:           9.59 bits
